# Trabalho Prático — Análise Exploratória e Pré-processamento de Dados
### Disciplina: Aprendizado de Máquina — IFMA Campus Coelho Neto
**Professor:** Bruno Vicente
**Dupla:** Rubens Dutra de Mesquita Filho e Pedro Alexandre
**Base de dados:** Bank Marketing (UCI Machine Learning Repository)
**Data:** Setembro/2026

---


## 0. Configuração inicial
Importação das bibliotecas utilizadas ao longo do notebook.

In [ ]:
# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import io

# Bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações gerais de exibição
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

print("Bibliotecas carregadas com sucesso.")

## 1. Definição do problema

**Contexto e domínio:** a base reúne dados de campanhas de telemarketing de uma instituição
bancária portuguesa. As campanhas consistiram em ligações telefônicas oferecendo a clientes a
adesão a um depósito a prazo (aplicação financeira). Cada instância representa um cliente
contatado, descrito por atributos demográficos, financeiros e características do próprio contato
telefônico.

**Atributo-alvo:** `y`, atributo categórico binário (`yes` = aderiu ao depósito, `no` = não
aderiu).

**Tipo de tarefa:** classificação binária.

**Relevância:** um modelo capaz de prever a adesão ao produto ajuda a instituição financeira a
priorizar quais clientes contatar, reduzindo o custo operacional de campanhas de telemarketing e
aumentando a taxa de conversão. Beneficia tanto o banco (eficiência de campanha) quanto o cliente
(menos contatos desnecessários).

**Observação metodológica importante:** o atributo `duration` (duração da ligação, em segundos)
só é conhecido **depois** que a ligação acontece — ou seja, não estaria disponível no momento em
que um modelo precisaria decidir se vale a pena ligar para o cliente. Isso é conhecido na
literatura como um caso de *data leakage* em potencial. Como este trabalho não treina modelo,
mantemos `duration` na análise exploratória (ele é informativo sobre o processo de contato), mas
registramos essa ressalva para deixar claro que, numa etapa futura de modelagem, esse atributo
exigiria tratamento especial.

## 2. Coleta e compreensão dos dados

**Origem:** UCI Machine Learning Repository — *Bank Marketing* (doada por S. Moro, P. Cortez e
P. Rita). Link: https://archive.ics.uci.edu/dataset/222/bank+marketing

**Licença:** Creative Commons Attribution 4.0 International (CC BY 4.0).

**Citação (ABNT):**
> MORO, S.; CORTEZ, P.; RITA, P. **Bank Marketing**. UCI Machine Learning Repository, 2012. DOI: https://doi.org/10.24432/C5K306. Disponível em: https://archive.ics.uci.edu/dataset/222/bank+marketing. Acesso em: 23 set. 2026.

**Artigo de referência:** MORO, S.; CORTEZ, P.; RITA, P. A data-driven approach to predict the
success of bank telemarketing. **Decision Support Systems**, v. 62, p. 22-31, 2014.

Usamos aqui a versão `bank-full.csv` (45.211 instâncias, 16 atributos + alvo), a versão completa
com o conjunto "clássico" de 17 colunas.

In [ ]:
# Carregamento dos dados diretamente da fonte oficial (garante reprodutibilidade)
# O UCI distribui esta base como um .zip contendo mais de um arquivo,
# por isso extraímos o bank-full.csv em memória antes de carregar no pandas.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

resposta = urllib.request.urlopen(url)
arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta.read()))

with arquivo_zip.open("bank-full.csv") as f:
    df = pd.read_csv(f, sep=";")

print(f"Dimensões da base: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
# Tipos de dados de cada coluna, conforme o pandas os interpretou
df.dtypes

**Interpretação:** o pandas reconheceu automaticamente 7 colunas numéricas (`age`, `balance`,
`day`, `duration`, `campaign`, `pdays`, `previous`) e 10 colunas categóricas/texto (`job`,
`marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome` e o
alvo `y`). Isso confirma a mistura de tipos exigida pelo trabalho — e, diferente da base anterior,
aqui **todos os nomes de coluna são documentados oficialmente pela fonte**, sem necessidade de
hipóteses de terceiros.

### Dicionário de dados

| Coluna | Tipo | Descrição | Valores possíveis / Unidade |
|---|---|---|---|
| `age` | Numérico (discreto) | Idade do cliente | anos |
| `job` | Categórico | Tipo de emprego | 12 categorias (admin, blue-collar, management, ... , unknown) |
| `marital` | Categórico | Estado civil | married, divorced, single (*divorced* inclui viúvo(a)) |
| `education` | Categórico | Nível de escolaridade | unknown, primary, secondary, tertiary |
| `default` | Categórico binário | Se o cliente possui crédito em situação de inadimplência | yes, no |
| `balance` | Numérico (contínuo) | Saldo médio anual da conta | euros |
| `housing` | Categórico binário | Se o cliente possui empréstimo habitacional | yes, no |
| `loan` | Categórico binário | Se o cliente possui empréstimo pessoal | yes, no |
| `contact` | Categórico | Tipo de contato usado na ligação | unknown, telephone, cellular |
| `day` | Numérico (discreto) | Dia do mês do último contato | 1-31 |
| `month` | Categórico | Mês do último contato | jan a dec |
| `duration` | Numérico (contínuo) | Duração do último contato | segundos (ver ressalva de *data leakage* na Etapa 1) |
| `campaign` | Numérico (discreto) | Número de contatos feitos nesta campanha para este cliente | contagem (inclui o último contato) |
| `pdays` | Numérico (discreto) | Dias desde o último contato de uma campanha anterior | dias (`-1` = cliente nunca contatado antes) |
| `previous` | Numérico (discreto) | Número de contatos feitos antes desta campanha | contagem |
| `poutcome` | Categórico | Resultado da campanha de marketing anterior | unknown, other, failure, success |
| `y` | **Categórico binário (alvo)** | Cliente aderiu ao depósito a prazo? | yes, no |

*Fonte: elaborado pelos autores, com base na documentação oficial do UCI (archive.ics.uci.edu/dataset/222/bank+marketing).*

> **Observação sobre a categoria "unknown":** a fonte oficial declara que a base **não possui
> valores ausentes (NaN)**. No entanto, quatro colunas categóricas (`job`, `education`, `contact`,
> `poutcome`) usam o valor `"unknown"` como uma categoria válida — que, na prática, funciona como
> um dado faltante disfarçado. Trataremos isso formalmente na Etapa 3 (diagnóstico de qualidade) e
> na Etapa 4 (pré-processamento), já que o pandas não vai contar essas ocorrências como `NaN`
> automaticamente.

In [ ]:
# Verificação de valores ausentes "tradicionais" (NaN)
print("Valores ausentes (NaN) por coluna:")
print(df.isnull().sum().sum(), "no total (esperado: 0, conforme documentação oficial)")

In [ ]:
# Verificação da categoria "unknown", que funciona como ausência disfarçada
colunas_com_unknown = ["job", "education", "contact", "poutcome"]

resumo_unknown = pd.DataFrame({
    "Ocorrências de 'unknown'": [(df[col] == "unknown").sum() for col in colunas_com_unknown],
    "Percentual (%)": [round((df[col] == "unknown").mean() * 100, 2) for col in colunas_com_unknown]
}, index=colunas_com_unknown)

resumo_unknown.sort_values("Ocorrências de 'unknown'", ascending=False)

**Interpretação:** a coluna `poutcome` tem a maior concentração de "unknown" (a maioria dos
clientes nunca havia sido contatada em campanha anterior, o que é coerente com `pdays = -1`).
`contact` também tem uma fração relevante de valores desconhecidos. Isso confirma que a base
atende ao critério de "dados reais com problemas de qualidade" exigido pelo trabalho, mesmo sem
ausentes no sentido técnico de `NaN`.

In [ ]:
# Verificação de linhas duplicadas
print(f"Linhas duplicadas: {df.duplicated().sum()}")

In [ ]:
# Distribuição inicial do atributo-alvo (visão rápida — a análise completa fica na Etapa 3)
df["y"].value_counts()


In [ ]:
# Percentual de cada classe do alvo — já dá pra ver o desbalanceamento
df["y"].value_counts(normalize=True).round(4) * 100

## 3. Análise Exploratória de Dados (AED)

Nesta etapa, exploramos estatisticamente e visualmente a base, atributo por atributo, sua relação
com o alvo (`y`) e os problemas de qualidade que ela apresenta.

### 3.1 Estatísticas descritivas

In [ ]:
# Estatísticas descritivas dos atributos numéricos
colunas_numericas = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
df[colunas_numericas].describe().T

**Interpretação:** logo se nota uma grande dispersão em `balance` (desvio-padrão de ~3.045,
contra uma média de ~1.362) e em `duration` e `campaign`, todos com o valor máximo muito distante
do 3º quartil — um forte indício de cauda longa e outliers, que investigamos com boxplots adiante.
Já `pdays` chama atenção pela mediana igual a `-1`: como vimos na Etapa 2, esse é o código-sentinela
para "nunca contatado antes", não um valor contínuo real — isso significa que estatísticas como
média e desvio-padrão de `pdays` (e também de `previous`) devem ser lidas com cautela, pois
misturam clientes "nunca contatados" com clientes que já tiveram contatos anteriores.

In [ ]:
# Estatísticas descritivas dos atributos categóricos
colunas_categoricas = ["job", "marital", "education", "default", "housing",
                        "loan", "contact", "month", "poutcome", "y"]
df[colunas_categoricas].describe()

**Interpretação:** o `describe()` de atributos categóricos mostra a categoria mais frequente
(`top`) e sua contagem (`freq`) para cada coluna. Vale destacar `poutcome`, cuja categoria mais
frequente é `unknown` (correspondente aos 81,7% de clientes nunca contatados antes, já visto na
Etapa 2), e `month`, cuja categoria mais frequente é `may` — indicando que maio concentrou boa
parte das campanhas de telemarketing dessa base.

### 3.2 Distribuição dos atributos numéricos

In [ ]:
# Histogramas de todos os atributos numéricos
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, coluna in enumerate(colunas_numericas):
    sns.histplot(df[coluna], bins=40, ax=axes[i], color="#4C72B0")
    axes[i].set_title(f"Distribuição de {coluna}")

# Remove os eixos vazios (sobraram 2, já que temos 7 colunas em uma grade 3x3)
for j in range(len(colunas_numericas), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Interpretação:** `age` se aproxima de uma distribuição normal levemente assimétrica à
direita, concentrada entre 30 e 50 anos. `balance`, `duration`, `campaign` e `previous` são
fortemente assimétricos à direita (a maioria dos clientes tem valores baixos, com uma cauda longa
de valores altos) — típico de variáveis financeiras e de contagem de eventos raros. `day` é
praticamente uniforme entre 1 e 31, coerente com contatos distribuídos ao longo do mês. `pdays`
mostra uma barra dominante em `-1` (o sentinela de "nunca contatado"), com uma cauda dispersa de
valores positivos — reforçando que essa coluna funciona quase como duas informações misturadas:
"foi contatado antes? (sim/não)" e, se sim, "há quantos dias".

In [ ]:
# Boxplots de todos os atributos numéricos, para identificar outliers
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, coluna in enumerate(colunas_numericas):
    sns.boxplot(x=df[coluna], ax=axes[i], color="#DD8452")
    axes[i].set_title(f"Boxplot de {coluna}")

for j in range(len(colunas_numericas), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Quantificação dos outliers de cada atributo numérico, pelo critério do IQR (intervalo interquartil)
resumo_outliers = []
for coluna in colunas_numericas:
    q1, q3 = df[coluna].quantile(0.25), df[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    qtd_outliers = ((df[coluna] < limite_inferior) | (df[coluna] > limite_superior)).sum()
    resumo_outliers.append({
        "Atributo": coluna,
        "Limite inferior (IQR)": round(limite_inferior, 2),
        "Limite superior (IQR)": round(limite_superior, 2),
        "Qtd. outliers": qtd_outliers,
        "% outliers": round(qtd_outliers / len(df) * 100, 2)
    })

pd.DataFrame(resumo_outliers).set_index("Atributo")

**Interpretação:** pelo critério do IQR, `pdays` e `previous` aparentam ter ~18% de
"outliers" — mas isso é enganoso: como a maioria dos clientes tem `pdays = -1` (sentinela) e
`previous = 0`, o próprio IQR fica colado nesses valores, fazendo qualquer contato anterior real
parecer outlier. Portanto, tratamos esse "outlier" como uma característica estrutural da coluna,
não como erro de dado.

Já `balance` (10,46% de outliers, com mínimo de -8.019 e máximo de 102.127) e `duration` (7,16%,
até 4.918 segundos ≈ 82 minutos) são outliers "de verdade" — saldos bancários e durações de
ligação extremos, mas plausíveis (existem clientes com saldo alto e ligações longas). `campaign`
(6,78%) reflete clientes contatados repetidamente na mesma campanha (até 63 vezes!). `age` tem
poucos outliers (1,08%) e `day` não tem nenhum, como esperado de um atributo com intervalo fixo
(1 a 31). Esses achados vão orientar as decisões de tratamento na Etapa 4.

### 3.3 Distribuição dos atributos categóricos

In [ ]:
# Gráficos de barra de todos os atributos categóricos (exceto o alvo, visto na seção 3.4)
colunas_categoricas_features = ["job", "marital", "education", "default",
                                  "housing", "loan", "contact", "month", "poutcome"]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, coluna in enumerate(colunas_categoricas_features):
    ordem = df[coluna].value_counts().index
    sns.countplot(y=df[coluna], order=ordem, ax=axes[i], color="#55A868")
    axes[i].set_title(f"Distribuição de {coluna}")
    axes[i].set_xlabel("Contagem")

plt.tight_layout()
plt.show()

**Interpretação:** `job` é dominado por `blue-collar` e `management`, mas está bem
distribuído entre 12 categorias, sem nenhuma concentração extrema. `marital` mostra maioria de
clientes casados (60%). `education` concentra-se em `secondary`. `default` (inadimplência) é
extremamente raro (só 815 de 45.211 clientes, ~1,8%) — um desbalanceamento próprio desse atributo,
não só do alvo. `housing` está relativamente equilibrado (55%/45%), mas `loan` mostra que a maioria
não tem empréstimo pessoal. `contact` é dominado por `cellular`, mas com 28,8% `unknown`. `month`
confirma a concentração em maio (13.766 contatos, quase 1/3 da base) — um possível viés temporal
que merece nota no relatório. `poutcome` reforça o que já vimos: mais de 80% dos casos são
"unknown" (cliente nunca contatado antes em campanha anterior).

### 3.4 Distribuição do atributo-alvo e verificação de desbalanceamento

In [ ]:
# Distribuição do atributo-alvo com percentuais
fig, ax = plt.subplots(figsize=(6, 5))
contagem_y = df["y"].value_counts()
percentual_y = df["y"].value_counts(normalize=True) * 100

barras = ax.bar(contagem_y.index, contagem_y.values, color=["#C44E52", "#55A868"])
for barra, pct in zip(barras, percentual_y.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 500,
            f"{pct:.1f}%", ha="center", fontweight="bold")

ax.set_title("Distribuição do atributo-alvo (y): aderiu ao depósito a prazo?")
ax.set_ylabel("Contagem de clientes")
plt.show()

**Interpretação:** a base é fortemente desbalanceada: 88,3% dos clientes **não** aderiram ao
depósito a prazo, contra apenas 11,7% que aderiram — uma proporção de aproximadamente 7,5 "não"
para cada "sim". Esse é exatamente o tipo de desbalanceamento que o trabalho pede para ser tratado
na Etapa 4 (com técnicas aplicadas somente no conjunto de treino, após o split). Esse
desbalanceamento também é esperado no domínio: campanhas de telemarketing costumam ter taxas de
conversão baixas.

### 3.5 Correlação entre atributos numéricos

In [ ]:
# Matriz de correlação (Pearson) entre os atributos numéricos
matriz_correlacao = df[colunas_numericas].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matriz_correlacao, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, ax=ax)
ax.set_title("Matriz de correlação — atributos numéricos")
plt.tight_layout()
plt.show()

**Interpretação:** as correlações entre os atributos numéricos são, de modo geral, fracas
(a maioria abaixo de 0,1 em módulo) — ou seja, não há problema relevante de multicolinearidade
nessa base. A única correlação que se destaca é entre `pdays` e `previous` (0,45): faz sentido,
já que ambas se referem ao histórico de contatos de campanhas anteriores, e quem tem `previous > 0`
necessariamente tem `pdays` diferente de `-1`. Há também uma correlação fraca-positiva entre `day`
e `campaign` (0,16), sugerindo que em determinados dias do mês houve mais contatos repetidos por
cliente.

### 3.6 Relação dos atributos com o atributo-alvo

In [ ]:
# Atributos numéricos vs. alvo: boxplots comparando as distribuições entre quem aderiu e quem não aderiu
colunas_numericas_relevantes = ["duration", "balance", "age", "previous"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, coluna in enumerate(colunas_numericas_relevantes):
    sns.boxplot(x="y", y=coluna, data=df, ax=axes[i], palette=["#C44E52", "#55A868"])
    axes[i].set_title(f"{coluna} por classe do alvo")

plt.tight_layout()
plt.show()

In [ ]:
# Médias dos atributos numéricos, separadas por classe do alvo
df.groupby("y")[colunas_numericas_relevantes].mean().round(2)

**Interpretação:** `duration` mostra a diferença mais clara entre as classes: clientes que
aderiram (`yes`) tiveram ligações com duração média de 537 segundos, contra 221 segundos entre os
que não aderiram — mais que o dobro. Isso é coerente com a ressalva de *data leakage* feita na
Etapa 1: ligações mais longas tendem a preceder uma resposta positiva, então esse atributo é
altamente informativo, mas não estaria disponível *antes* da ligação acontecer. `previous` também
é maior entre quem aderiu (1,17 contatos anteriores, em média, contra 0,50), sugerindo que clientes
já contatados em campanhas passadas respondem melhor. `balance` e `age` mostram diferenças bem mais
sutis entre os grupos, indicando menor poder discriminativo isoladamente.

In [ ]:
# Atributos categóricos vs. alvo: taxa de adesão (%) por categoria
colunas_categoricas_relevantes = ["job", "poutcome", "education", "housing", "contact"]

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for i, coluna in enumerate(colunas_categoricas_relevantes):
    taxa_adesao = (df.groupby(coluna)["y"]
                     .apply(lambda s: (s == "yes").mean() * 100)
                     .sort_values(ascending=False))
    sns.barplot(x=taxa_adesao.values, y=taxa_adesao.index, ax=axes[i], color="#4C72B0")
    axes[i].set_title(f"Taxa de adesão (%) por {coluna}")
    axes[i].set_xlabel("% que aderiu (y = yes)")

plt.tight_layout()
plt.show()

**Interpretação:** `poutcome` é o atributo categórico com maior poder discriminativo:
clientes cuja campanha anterior teve sucesso (`success`) aderem em 64,7% dos casos, contra apenas
9,2% entre os que nunca foram contatados antes (`unknown`) — uma diferença enorme. `job` também
mostra padrão claro: `student` (28,7%) e `retired` (22,8%) aderem muito mais que `blue-collar`
(7,3%) e `entrepreneur` (8,3%) — plausível, já que estudantes e aposentados costumam ter mais
disponibilidade e menos compromissos financeiros de curto prazo. `education` mostra que quanto
maior a escolaridade, maior a taxa de adesão (`tertiary`: 15,0% vs. `primary`: 8,6%). `housing`
mostra que clientes **sem** empréstimo habitacional aderem mais (16,7% vs. 7,7%) — plausível, já
que menos comprometimento financeiro deixa mais espaço para uma nova aplicação. Por fim, `contact`
mostra que contatos com tipo `unknown` têm a menor taxa de adesão (4,1%) — reforçando que esse
"unknown" pode não ser aleatório, e sim concentrado em contatos de pior qualidade ou mais antigos.

### 3.7 Diagnóstico de qualidade dos dados (consolidação)

Resumo dos problemas de qualidade identificados ao longo da AED, que serão tratados formalmente na
Etapa 4 (Pré-processamento):

In [ ]:
# Consolidação dos problemas de qualidade encontrados
print("1) Valores ausentes (NaN):", df.isnull().sum().sum(), "(confirmado: a fonte não usa NaN)")
print("2) Linhas duplicadas:", df.duplicated().sum())
print("3) Categoria 'unknown' (ausência disfarçada) em: job, education, contact, poutcome")
print(f"4) Saldos bancários negativos (balance < 0): {(df['balance'] < 0).sum()} "
      f"({(df['balance'] < 0).mean() * 100:.2f}%)")
print(f"5) Ligações com duração zero: {(df['duration'] == 0).sum()} "
      f"(possível inconsistência de registro)")
print(f"6) Desbalanceamento do alvo: {df['y'].value_counts(normalize=True)['no']*100:.1f}% "
      f"'no' vs. {df['y'].value_counts(normalize=True)['yes']*100:.1f}% 'yes'")

**Interpretação final da AED:** a base **não tem valores ausentes formais nem duplicados**,
mas apresenta problemas de qualidade reais e relevantes: (1) ausência disfarçada via `"unknown"`
em 4 colunas categóricas; (2) outliers genuínos em `balance`, `duration` e `campaign`; (3) saldos
bancários negativos (8,33% da base) — plausíveis (contas com cheque especial/overdraft), mas que
precisam de atenção; (4) 3 registros com `duration = 0`, que são fisicamente estranhos (uma
ligação que "durou" zero segundos não deveria gerar um resultado de campanha) e candidatos a
remoção ou investigação na Etapa 4; e (5) forte desbalanceamento de classes (88,3%/11,7%). Todos
esses pontos serão tratados, com justificativa técnica, na próxima etapa.